In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rajeevgautam335","key":"98d2da5dc5e5dce52c99688a5b0356d1"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d biaiscience/dogs-vs-cats

Dataset URL: https://www.kaggle.com/datasets/biaiscience/dogs-vs-cats
License(s): DbCL-1.0
 99% 810M/817M [00:11<00:00, 35.6MB/s]
100% 817M/817M [00:11<00:00, 76.9MB/s]


In [ ]:
import zipfile

with zipfile.ZipFile("dogs-vs-cats.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [ ]:
import os
os.listdir("data")

['train', 'test']

In [ ]:
import os

train_dir = "data/train"
test_dir = "data/test"

print("Train folders:", os.listdir(train_dir))
print("Test folders:", os.listdir(test_dir))

Train folders: ['train']
Test folders: ['test']


In [ ]:
import os, shutil, random

base_dir = "dataset"
train_dir = base_dir + "/train"
val_dir = base_dir + "/val"

os.makedirs(train_dir + "/cats", exist_ok=True)
os.makedirs(train_dir + "/dogs", exist_ok=True)
os.makedirs(val_dir + "/cats", exist_ok=True)
os.makedirs(val_dir + "/dogs", exist_ok=True)

In [ ]:
images = os.listdir("data/train/train")
random.shuffle(images)

split = int(0.8 * len(images))
train_images = images[:split]
val_images = images[split:]

def move_images(image_list, target_dir):
    for img in image_list:
        label = "cats" if img.startswith("cat") else "dogs"
        shutil.copy(
            "data/train/train/" + img,
            target_dir + "/" + label + "/" + img
        )

move_images(train_images, train_dir)
move_images(val_images, val_dir)

In [ ]:
images = os.listdir("data/train/train")
random.shuffle(images)

split = int(0.8 * len(images))
train_images = images[:split]
val_images = images[split:]

def move_images(image_list, target_dir):
    for img in image_list:
        label = "cats" if img.startswith("cat") else "dogs"
        shutil.copy(
            "data/train/train/" + img,
            target_dir + "/" + label + "/" + img
        )

move_images(train_images, train_dir)
move_images(val_images, val_dir)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(base_model.input, output)

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
rain_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=40,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   fill_mode='nearest')

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

Found 23982 images belonging to 2 classes.
Found 8982 images belonging to 2 classes.


In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 335s 435ms/step - accuracy: 0.9521 - loss: 0.1233 - val_accuracy: 0.9820 - val_loss: 0.0459
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 302s 403ms/step - accuracy: 0.9686 - loss: 0.0779 - val_accuracy: 0.9862 - val_loss: 0.0383
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 301s 401ms/step - accuracy: 0.9702 - loss: 0.0723 - val_accuracy: 0.9863 - val_loss: 0.0368
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 302s 402ms/step - accuracy: 0.9717 - loss: 0.0654 - val_accuracy: 0.9791 - val_loss: 0.0553
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 302s 403ms/step - accuracy: 0.9728 - loss: 0.0667 - val_accuracy: 0.9840 - val_loss: 0.0408
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 307s 409ms/step - accuracy: 0.9734 - loss: 0.0670 - val_accuracy: 0.9859 - val_loss: 0.0358
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 302s 403ms/step - accuracy: 0.9740 - loss: 0.0595 - val_accuracy: 0.9847 - val_loss: 0.0387
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 303s 403ms/step - accuracy: 0.9776 -

In [ ]:
model.save("cats_vs_dogs_transfer_learning.h5")